In [ ]:
import cv2
import numpy as np
from collections import deque

# Store tracked points
pts = deque(maxlen=512)

# HSV range for blue color
Lower_green = np.array([110, 50, 50])
Upper_green = np.array([130, 255, 255])

# Morphological kernel
kernel = np.ones((5, 5), np.uint8)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, img = cap.read()

    if not ret:
        break

    # Flip frame horizontally
    img = cv2.flip(img, 1)

    # Convert BGR to HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Create mask for blue color
    mask = cv2.inRange(hsv, Lower_green, Upper_green)

    # Remove noise
    mask = cv2.erode(mask, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)

    # Extract blue object
    res = cv2.bitwise_and(img, img, mask=mask)

    # Find contours
    cnts, hierarchy = cv2.findContours(
        mask.copy(),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    center = None

    if len(cnts) > 0:

        # Largest contour
        c = max(cnts, key=cv2.contourArea)

        # Minimum enclosing circle
        ((x, y), radius) = cv2.minEnclosingCircle(c)

        # Moments
        M = cv2.moments(c)

        if M["m00"] != 0:
            center = (
                int(M["m10"] / M["m00"]),
                int(M["m01"] / M["m00"])
            )

            if radius > 5:

                # Draw circle around marker
                cv2.circle(
                    img,
                    (int(x), int(y)),
                    int(radius),
                    (0, 255, 255),
                    2,
                )

                # Draw center
                cv2.circle(
                    img,
                    center,
                    5,
                    (0, 0, 255),
                    -1,
                )

                pts.appendleft(center)

    # Draw movement
    for i in range(1, len(pts)):

        if pts[i - 1] is None or pts[i] is None:
            continue

        thick = int(np.sqrt(len(pts) / float(i + 1)) * 2.5)

        cv2.line(
            img,
            pts[i - 1],
            pts[i],
            (0, 0, 255),
            thick,
        )

    # Display windows
    cv2.imshow("Frame", img)
    cv2.imshow("Mask", mask)
    cv2.imshow("Result", res)

    key = cv2.waitKey(1) & 0xFF

    # Press q to quit
    if key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()